In [1]:
!pip install -q langchain langchain-groq gspread

In [3]:
import os
from google.colab import userdata, auth
os.environ["GROQ_API_KEY"] = userdata.get("GROQ_API_KEY")

from langchain_groq import ChatGroq
llm = ChatGroq(model="openai/gpt-oss-20b", temperature=0.2, max_retries=3)

auth.authenticate_user()
import gspread
from google.auth import default
creds, _ = default()
gc = gspread.authorize(creds)
sh = gc.create("Smart Mobile Store Orders")      # run once; next time use gc.open("Smart Mobile Store Orders")
ws = sh.sheet1
ws.append_row(["name", "mobile_model", "qty", "date", "storage", "total"])
print("Sheet ready:", sh.url)

Sheet ready: https://docs.google.com/spreadsheets/d/1BjU9VUWJBtJ6zDo5VhBYjtPWeTwNJqUHsvS5YPtW8UA


In [4]:
from typing import Literal
from pydantic import BaseModel, Field
from langchain.tools import tool

MENU = {
    "iphone 15": 55000,
    "samsung galaxy s24": 45000,
    "oneplus 12": 60000,
    "redmi note 13": 18000
}
HOURS = {
    "monday": "9 am to 8 pm",
    "tuesday": "9 am to 8 pm",
    "wednesday": "9 am to 8 pm",
    "thursday": "9 am to 8 pm",
    "friday": "9 am to 8 pm",
    "saturday": "9 am to 8 pm",
    "sunday": "10 am to 6 pm"
}
class Order(BaseModel):
    """A mobile phone order taken from a customer's chat message."""
    name: str = Field(description="Customer's first name")
    item: Literal[
        "iphone 15",
        "samsung galaxy s24",
        "oneplus 12",
        "redmi note 13"
    ] = Field(description="Mobile phone model")
    qty: int = Field(description="How many mobile phones", ge=1)
    date: str = Field(description="Purchase date as the customer said it")
    storage: Literal["128GB", "256GB", "512GB"] = Field(
        description="Mobile storage variant selected by the customer"
    )

In [5]:
@tool
def get_price(item: str, qty: int = 1) -> str:
    """Get the total price in rupees for a quantity of one mobile phone.
    Args:
        item: Mobile model, e.g. "iphone 15", "samsung galaxy s24",
              "oneplus 12" or "redmi note 13"
        qty: How many mobile phones the customer wants
    """
    price = MENU.get(item.lower().strip())
    if price is None:
        return f"'{item}' is not available. We have: {', '.join(MENU)}."

    return f"{qty} x {item} = Rs {price * qty}"

@tool
def is_open(day: str) -> str:
    """Check the mobile shop's opening hours for a day of the week, e.g. 'sunday'."""
    hours = HOURS.get(day.lower().strip())
    return f"Open {hours} on {day}." if hours else f"'{day}' is not a day of the week."

@tool
def log_order(name: str, item: str, qty: int, date: str, storage: str) -> str:
    """Save a confirmed mobile phone order to the Orders sheet.
    Use only when the customer has given their name, mobile model,
    quantity, purchase date and storage variant."""
    try:
        order = Order(
            name=name,
            item=item.lower().strip(),
            qty=qty,
            date=date,
            storage=storage
        )
    except Exception as e:
        return f"Could not save the order: {e}. Ask the customer to correct it."
    total = MENU[order.item] * order.qty
    ws.append_row([
        order.name,
        order.item,
        order.qty,
        order.date,
        order.storage,
        total
    ])
    return f"Saved: {order.qty} x {order.item} ({order.storage}) for {order.name} on {order.date}, Rs {total}."

In [6]:
from langchain.agents import create_agent
SYSTEM = """ROLE:
You are the friendly order assistant for Smart Mobile Store.
FACTS:
iPhone 15 price Rs 55000.
Samsung Galaxy S24 price Rs 45000.
OnePlus 12 price Rs 60000.
Redmi Note 13 price Rs 18000.
Orders need 24 hours' notice.

RULES:
Always use get_price for prices and is_open for shop timings.
Never guess prices or timings.
Before log_order, make sure you have customer name, mobile model,
quantity, purchase date and storage variant.
Ask the customer if any required information is missing.
Save the order only after customer confirmation.
After saving, read the order back with the total.

FORMAT:
Under 60 words, friendly, English or Telugu matching the customer.
"""
agent = create_agent(
    model=llm,
    tools=[get_price, is_open, log_order],
    system_prompt=SYSTEM
)
history = []

def chat(text: str) -> str:
    global history
    try:
        result = agent.invoke({
            "messages": history + [
                {"role": "user", "content": text}
            ]
        })
        history = result["messages"]
        return history[-1].content
    except Exception as e:
        print("Agent error:", e)
        return "Sorry, I'm a little busy. Please try again in a minute."
print(chat("Are you open on Sunday? How much are 2 iPhone 15 phones?"))

Yes, we’re open Sunday from 10 am to 6 pm.  
Two iPhone 15s cost Rs 110,000.  
Would you like to place an order? (We’ll need your name, quantity, purchase date, storage variant, and a 24‑hour notice.)


In [7]:
print(chat("I'm Ravi. Please order 2 iPhone 15 phones with 128GB storage for 24 Dec."))

✅ Order confirmed!  
2 × iPhone 15 (128 GB) for Ravi on 24 Dec – Total Rs 110,000.  
We’ll see you on the 24th. Let me know if you need anything else!
